In [6]:
import csv
from itertools import dropwhile

import pandas as pd
from sqlalchemy.cyextension.resultproxy import rowproxy_reconstructor

## READ FILE

In [7]:
def read_txt_with_delimiter(filename, delimiter):
    data = []
    with open(filename, 'r', newline='', encoding='utf-8') as file:
        # Create a reader object with the specified delimiter
        reader = csv.reader(file, delimiter=delimiter)
        for row in reader:
            data.append(row)
    dataframe = pd.DataFrame(data[1:], columns=data[0])  # Assuming the first row contains headers
    return dataframe

In [14]:
file_name = 'chunks/member_details_errors_chunk_1.txt'
delimiter = '|'
df = read_txt_with_delimiter(file_name, delimiter)
df.head()

,CIN,Segment,Membership_status,title_code,first_name,last_name,gender_code,date_of_birth,address_line_1,address_line_2,town_city,state_region,post_code,country_code,email_address,language_code,validation_errors
0,1727923634,CP9CB,1,Doctor,Rana Mahmoud Abdelshafi Ali,El Azab,F,1970-10-13,16,Oaks Park,Rough Common Canterbury,Canterbury Kent,CT2 9DP,GB,ranaelazab_2000@yahoo.co.uk,en-gb,Duplicate CIN value '1727923634'
1,1727923634,UJAAA,0,Doctor,Rana Mahmoud Abdelshafi Ali,El Azab,F,1970-10-13,16,Oaks Park,Rough Common Canterbury,Canterbury Kent,CT2 9DP,GB,ranaelazab_2000@yahoo.co.uk,en-gb,Duplicate CIN value '1727923634'
2,1908531525,UJADF,0,Sir,Francis Nicholas Fraser,Pearson,M,1943-08-28,9,Upper Addison Gardens,London,London,W14 8AL,GB,snickpearson@gmail.com,en-gb,Duplicate CIN value '1908531525'
3,1814751890,UJACF,0,Doctor,Emily Claire,Duncan,F,1980-03-30,11,Madeira Road,Clevedon,Clevedon Avon,BS21 7TJ,GB,emilyduncan@doctors.org.uk,en-gb,Duplicate CIN value '1814751890'
4,1814751890,UJACF,0,Doctor,Emily Claire,Duncan,F,1980-03-30,11,Madeira Road,Clevedon,Clevedon Avon,BS21 7TJ,GB,emilyduncan@doctors.org.uk,en-gb,Duplicate CIN value '1814751890'


In [8]:
file_name = 'member_details.txt'
delimiter = '|'
df = read_txt_with_delimiter(file_name, delimiter)
df.head()

,CIN,Segment,Membership_status,title_code,first_name,last_name,gender_code,date_of_birth,address_line_1,address_line_2,town_city,state_region,post_code,country_code,email_address,language_code
0,1726507343,CP9AB,1,DR,MARK CHARLES KEMPSTER,RANCE,M,1975-09-15,6,Chalmers Way,Twickenham,Twickenham,TW1 1QG,GB,thaliaandmark@gmail.com,en-gb
1,1207579122,,0,Prof,Paul Richard,Blackledge,M,1967-08-02,53,Ridgeway,Leeds,Leeds,LS8 4DD,GB,p.blackledge@aol.com,en-gb
2,1727923634,,0,Doctor,Rana Mahmoud Abdelshafi Ali,El Azab,F,1970-10-13,16,Oaks Park,Rough Common Canterbury,Canterbury Kent,CT2 9DP,GB,ranaelazab_2000@yahoo.co.uk,en-gb
3,1727923634,CP9CB,1,Doctor,Rana Mahmoud Abdelshafi Ali,El Azab,F,1970-10-13,16,Oaks Park,Rough Common Canterbury,Canterbury Kent,CT2 9DP,GB,ranaelazab_2000@yahoo.co.uk,en-gb
4,1727923634,UJAAA,0,Doctor,Rana Mahmoud Abdelshafi Ali,El Azab,F,1970-10-13,16,Oaks Park,Rough Common Canterbury,Canterbury Kent,CT2 9DP,GB,ranaelazab_2000@yahoo.co.uk,en-gb


In [9]:
df.columns

Index(['CIN', 'Segment', 'Membership_status', 'title_code', 'first_name',
       'last_name', 'gender_code', 'date_of_birth', 'address_line_1',
       'address_line_2', 'town_city', 'state_region', 'post_code',
       'country_code', 'email_address', 'language_code'],
      dtype='str')

In [10]:
len(df.columns)

16

In [11]:
file_name = 'hsbc_million_members.txt'
delimiter = '|'
df2 = read_txt_with_delimiter(file_name, delimiter)
df2.head(2)

,CIN,Segment,New_joiner,title_code,first_name,last_name,gender_code,date_of_birth,address_line_1,address_line_2,town_city,state_region,post_code,country_code,email_address,language_code,member_status
0,2780100523,1,1,,Noah,O'Brien,,,655 Main Dr,,Dublin,NC,Y2A 3K0,,noah.obrien@example.com,BCP 47,1
1,5296645123,1,0,MS,Jessica,Mehta,M,1966-06-02,400 River Terrace,,Durban,QC,38836,FR,jessica514@outlook.com,BCP 47,1


In [12]:
df2.columns

Index(['CIN', 'Segment', 'New_joiner', 'title_code', 'first_name', 'last_name',
       'gender_code', 'date_of_birth', 'address_line_1', 'address_line_2',
       'town_city', 'state_region', 'post_code', 'country_code',
       'email_address', 'language_code', 'member_status'],
      dtype='str')

In [13]:
len(df2.columns)

17

# WRANGLING

In [275]:
#drop duplicate rows
df.drop_duplicates(inplace=True)

## MANDATORY FIELDS

In [276]:
#mandatory fields in supplied file
required_fields={
    "CIN": True,
    "Segment": False,
    "first_name": True,
    "last_name": True,
    "address_line_1": True,
    "post_code": False,
    "country_code": False,
    "email_address": False,
    "date_of_birth": False,
    "title_code": False,
    "gender_code": False,
    "Membership_status": True
  }

# Validate Fields

## DROP NA MANDATORY FIELDS

In [277]:
# Drop na Values if they are part of mandatory fields and log the count of dropped records
mandatory_cols = [
    col for col, is_required in required_fields.items()
    if is_required
]

df = df.dropna(subset=mandatory_cols)


## Validate Email

In [280]:
regex_email = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
df['email_valid'] = df['email_address'].str.match(regex_email)
df = df[df['email_valid'].notna() & df['email_valid'] == True]
df.drop('email_valid', axis=1, inplace=True)

# Validate Postcode

In [281]:
postcode_regex = r'^[A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2}$'
df['postcode_valid'] = df['post_code'].str.match(postcode_regex)
df = df[df['postcode_valid'].notna() & (df['postcode_valid'] == True)]
df.drop('postcode_valid', axis=1, inplace=True)

In [263]:
df.head(2)

,CIN,Segment,Membership_status,title_code,first_name,last_name,gender_code,date_of_birth,address_line_1,address_line_2,town_city,state_region,post_code,country_code,email_address,language_code
0,1726507343,CP9AB,1,DR,MARK CHARLES KEMPSTER,RANCE,M,1975-09-15,6,Chalmers Way,Twickenham,Twickenham,TW1 1QG,GB,thaliaandmark@gmail.com,en-gb
1,1207579122,,0,Prof,Paul Richard,Blackledge,M,1967-08-02,53,Ridgeway,Leeds,Leeds,LS8 4DD,GB,p.blackledge@aol.com,en-gb
